# RRT* (Rapidly-exploring Random Tree Star) Path Planning

### What This Script Does
This notebook builds a ground-up simulation of the RRT* path-planning algorithm. It sets up a 2D environment with a starting coordinate and a goal coordinate. When executed, it uses `matplotlib` and IPython's display tools to create a live, frame-by-frame animation of the algorithm searching the space, building its tree, optimizing its branches, and finally highlighting the most efficient path it found.

### How the RRT* Algorithm Works
Standard RRT grows a random web of lines until it stumbles into the goal, often resulting in highly inefficient, jagged paths. **RRT*** adds an optimization step ("rewiring") that constantly smooths and shortens the path as it explores.

Here is the loop it follows:
1. **Sample:** It picks a random `(x, y)` coordinate on the map.
2. **Nearest:** It scans its existing tree to find the node closest to that random point.
3. **Steer:** It grows a new branch from that closest node toward the random point (by a fixed `step_size`).
4. **Near Neighbors:** It draws a search radius around this newly created branch and finds all other existing nodes within that circle.
5. **Choose Best Parent:** Instead of blindly connecting the new branch to the node that spawned it, it calculates the total distance from the *start* point through all those neighbors. It connects to the neighbor that yields the lowest total distance.
6. **Rewire (The "Star"):** It looks at those neighbors again. If any of those neighbors could get a shorter path back to the start by routing *through* our new branch, the algorithm deletes their old connection and "rewires" them to the new branch. This continuously straightens the tree.

### Setup, Imports, and Node Data Structure
Here we import the math and visualization libraries. We also define the `Node` class, which is the fundamental building block of our tree. Each Node represents a physical point in space and holds the history of how the algorithm reached it.


In [ ]:
# install required packages if not already present
%pip install numpy

import math
import random
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import clear_output

class Node:
    """
    Represents a single point in the RRT* tree.

    structure example:
    if a node is at coordinates (10,5), costs 15 units of distance from the start
    and is connected to a previous node called node_A:
    node.x=10
    node.y=5
    node.parent=node_A
    node.cost=15
    """
    def __init__(self,x,y):
        self.x=x
        self.y=y
        self.parent=None # node that comes before this node in the path
        self.cost=0.0 #total distance from the start node to this node
        

### The RRT* Class Engine
This cell contains the entire logic of the RRT* algorithm. 

The `plan()` method is the main loop that orchestrates generating points, steering, evaluating parents, rewiring, and drawing the live plot.

In [ ]:
class RRTStar:
    # start and goal are tuples of (x,y) coordinates
    # area_size is a list of [x_min,x_max,y_min,y_max] defining the bounds of the search area
    # step_size is the maximum distance between nodes in the tree
    # search_radius is the radius within which to look for nearby nodes when rewiring 
    # max_iter is the maximum number of iterations to run the algorithm
    def __init__(self,start,goal,area_size,step_size=2.0,search_radius=10.0,max_iter=300):
        #convert the raw (x,y) into Node objects
        self.start=Node(start[0],start[1])
        self.goal=Node(goal[0],goal[1])

        #area_size format is[x_min,x_max,y_min,y_max]
        self.area_size=area_size

        #how far a new branch grows in one step
        self.step_size=step_size

        #how wide of a radius to look for neighbors during optimization
        self.search_radius=search_radius

        #how many random points to generate before giving up
        self.max_iter=max_iter

        #main list storing every single node in the tree
        self.nodes=[self.start]

    def get_random_node(self):
        #generate a random (x,y) within the area
        #10% of the time, cheat and return the goal to bias the search towards it
        if random.randint(0,100)>10:
            #generate a random float between min/max for x and y
            rnd=Node(random.uniform(self.area_size[0],self.area_size[1]),
                     random.uniform(self.area_size[2],self.area_size[3]))
            return rnd
        else:

            return Node(self.goal.x,self.goal.y)

    def get_nearest_node(self,rnd_node):
        #finds the index of the node in self.nodes that is closest to rnd_node

        #calculate the euclidean distance from rnd_node to every node in self.nodes
        #skip the square root for efficiency since we only care about relative distances
        dlist=[(node.x-rnd_node.x)**2+(node.y-rnd_node.y)**2 for node in self.nodes]

        #return the index of the smallest distance
        return dlist.index(min(dlist))